# Prefill: TTFT vs prompt length

## Performance of each phase of an LLM, starting with prefill

- **Prefill (this benchmark):** n prompts multiplied with weight matrices; builds the KV cache. Weights ~ n·d², attention ~ n²·d. Compute-bound.
- **Decode (TODO):** one token per step, re-reading weights and the whole KV cache. Memory-bandwidth-bound.
- **Hardware considerations:** which GPU, which precision, memory requirements, how much context to allow.

### Investigating prefill: where the KV cache is built and attention grows as n²

1. **Signal:** headlines about one ticker, and one question about it.
2. **Noise:** add 50 to 1,000 unrelated headlines to lengthen the prompt.
3. **Measure:** time to first token; 3 repeats, cache off.

First, naive version: all headlines and any question gave poor answers and no control over length.

## TTFT scales with prompt length non-linearly

Qwen3-4B (4-bit) on vllm-mlx · Apple M1, 8 GB · median of 3 repeats per point · prefix cache off, unique prompt per request.

| Ticker | Signal headlines | Noise headlines | Data | Config |
|---|---|---|---|---|
| WMT | 10 | 50, 100, 200, 300, 500, 1000 | [`results/ttft-prefill-baseline.csv`](results/ttft-prefill-baseline.csv) | [`configs/ttft-prefill-baseline.yaml`](configs/ttft-prefill-baseline.yaml) |
| AMZN | 7 | 50, 100, 200, 300, 500, 750 | [`results/ttft-prefill-baseline-amzn.csv`](results/ttft-prefill-baseline-amzn.csv) | [`configs/ttft-prefill-baseline-amzn.yaml`](configs/ttft-prefill-baseline-amzn.yaml) |

AMZN at 1,000 noise headlines returned no tokens, so its sweep stops at 750.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RUNS = {"WMT": "results/ttft-prefill-baseline.csv", "AMZN": "results/ttft-prefill-baseline-amzn.csv"}


def medians(path):
    df = pd.read_csv(path).dropna(subset=["ttft_s"])
    return df[df.ok].groupby("noise_headlines")[["prompt_tokens", "ttft_s"]].median()


def fit(m):
    n, t = m.prompt_tokens.to_numpy(), m.ttft_s.to_numpy()
    return np.linalg.lstsq(np.c_[n, n**2] / t[:, None], np.ones_like(t), rcond=None)[0]


meds = {k: medians(p) for k, p in RUNS.items()}
fits = {k: fit(m) for k, m in meds.items()}
for k, (b, c) in fits.items():
    n, t = meds[k].prompt_tokens, meds[k].ttft_s
    print(f"{k:4}  t ≈ {b * 1e3:.2f} ms·n + {c * 1e6:.3f} µs·n²   n* = b/c ≈ {b / c:,.0f}   {n.iloc[0] / t.iloc[0]:.0f} → {n.iloc[-1] / t.iloc[-1]:.0f} tokens/s")

b, c = fits["WMT"]
x = np.linspace(0, 1.05 * meds["WMT"].prompt_tokens.max(), 200)
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(x, b * x + c * x**2, label="fit (WMT)")
ax.plot(x, b * x, "--", label="if every token cost the same")
ax.plot(meds["WMT"].prompt_tokens, meds["WMT"].ttft_s, "ko", label="WMT, median of 3")
ax.plot(meds["AMZN"].prompt_tokens, meds["AMZN"].ttft_s, "s", mfc="none", mec="C3", ms=9, label="AMZN, median of 3")
ax.set(xlabel="Prompt length (tokens)", ylabel="Time to first token (s)", xlim=(0, None), ylim=(0, None))
ax.grid(alpha=0.3)
ax.legend();

## Check: prefill cost depends on token count, not content

The AMZN run swaps the signal: 7 Amazon headlines instead of 10 Walmart ones, about 49 more tokens. The noise headlines are the same in both runs, so over 97% of the prompt text is shared. The runs were a week apart.

- **Fits agree:** both tickers give the same coefficients to within 0.5% (printed above).
- **AMZN and WMT curves co-incide**
- **The ~49-token gap between AMZN and WMT is too small to resolve.** It is worth about 0.3% of TTFT. Run-to-run drift between two separate WMT runs is 0.2–2%, so the per-level AMZN − WMT gaps (0.21–0.44 s) do not track it. Interleaving tickers in one run would remove the drift.

Qwen3-4B is a dense model, so the FLOPs depend only on token count. A stronger test of content-independence would vary the noise headlines too.

In [ ]:
b, c = fits["WMT"]
amzn = meds["AMZN"].assign(wmt_fit_s=lambda m: b * m.prompt_tokens + c * m.prompt_tokens**2)
amzn["diff_%"] = 100 * (amzn.ttft_s / amzn.wmt_fit_s - 1)
amzn.round(2)

## Prefill (compute bound) vs decode (memory bound)

| | Prefill: reading the prompt | Decode: writing the answer |
|---|---|---|
| Input | n prompt tokens, all at once | 1 new token |
| Attention (each of 36 layers) | every token vs every earlier one: n² | new token vs all n cached: n |
| Weight multiplies | same cost per token: n | all weights re-read for one token |
| KV cache (keys and values of every token so far) | writes one entry per prompt token | read in full each step, one entry added |
| Output | first token · time to first token | next token · repeat until done |

Simplified decoder-only transformer, reference Vaswani et al., “Attention Is All You Need”, NeurIPS 2017 ([arXiv:1706.03762](https://arxiv.org/abs/1706.03762)). Layer count is Qwen3-4B’s.

## Prefill and decode stress different hardware

| | Prefill: reading the prompt | Decode: writing the answer |
|---|---|---|
| Tokens per step | all n prompt tokens at once | one |
| Weight multiplies | matrix × matrix: each weight loaded once, used by n tokens | matrix × vector: every weight loaded again for each token |
| Attention | every token against every earlier one: grows as n² | the new token against n cached keys: grows as n |
| KV cache | written, one entry per prompt token: grows as n | read in full every step, one entry appended |
| Bottleneck | compute (FLOP/s) | memory bandwidth (bytes/s) |
| What users see | time to first token | milliseconds per output token |
| GPU feature that helps | tensor-core throughput, FP8/FP4 | HBM bandwidth (H200 about 4.8 TB/s) |

Measured so far: prefill. Decode sweep next, same harness.